# M3: Feature Vectors Representation

This notebook documents an experiment with feature vector representations extracted using **Meta's DINOv3 open-source model**.

<img src="https://miro.medium.com/v2/resize:fit:1400/1*y3lPsbCSzNJ7VPuerUdBmA.png" alt="DINOv2 architecture diagram" width="60%">

The primary goal is to explore and understand the characteristics of these high-dimensional feature vectors when applied to a custom dataset.

Key objectives include:

* Extracting deep features for each sample in the dataset.
* Analyzing class separability by calculating the **mean feature vector** for each distinct class.
* Visualizing the relationships (e.Example, proximity and distance) between these class-level representations to see which classes cluster together.
* Applying **Principal Component Analysis (PCA)** to reduce dimensionality and further inspect the feature space.

Through these experiments, we aim to gain insights into how the DINOv2 model represents the different classes and the overall structure of our data.

## 1. About the DINOv3 Model

DINOv3 (short for **DI**stillation of knowledge with **NO** labels v3) is a foundational model developed by Meta AI.

Its key characteristic is that it learns powerful, general-purpose visual features using **self-supervised learning**. Unlike traditional models that require massive datasets with human-provided labels (e.g., "cat," "dog"), DINOv3 learns by analyzing the relationships between different parts of unlabeled images. It was trained on a curated dataset of 142 million images.

This approach, which uses a "student-teacher" architecture, allows the model to produce high-performance features that can be used directly for various downstream tasks (like classification, segmentation, or depth estimation) often without needing to be fine-tuned.

<img src="https://diplo-media.s3.eu-central-1.amazonaws.com/2025/06/Meta-AI-Discovery-pop-up-Share-1024x620.jpg" width="60%">


### **1.1 Importing**
To obtain our DINOv3 model, we will use [HuggingFace](https://huggingface.co/docs/transformers/main/model_doc/dinov3) and leverage the `transformers` library. The simplest approach is to use a pipeline, which provides the model in *inference* mode. This ensures that Dropout layers are turned off, resulting in stable and consistent feature vectors.

In [ ]:
#To install the needed libraries
!pip install transformers torch

In [7]:
import torch
import numpy as np
import torchvision
from transformers import pipeline
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from transformers import AutoImageProcessor, AutoModel
from tqdm.auto import tqdm
import h5py

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### *1.2 Logging in to Hugging Face*
To import this modul logging in and approval to DinoV3 is required


<img src="https://adictosaltrabajo.com/wp-content/uploads/2023/05/hf-logo-with-title.png" width="60%">

In [3]:
#required log in to hugging face
!pip install huggingface_hub -q

In [4]:
from huggingface_hub import login
login()

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
HDF5_FILE = "dino_features.h5"
print(f"Loading model: {model_name}")

model = AutoModel.from_pretrained(model_name).to(device).eval()

Loading model: facebook/dinov3-vits16-pretrain-lvd1689m


### 1.3 No Hugging face
If you are not fortunate enough to get access rights to `DinoV3` like me. We will make use of the DinoV2Model provided for everyone using Pytorch.

There are two options for importing:
- Official Pytorch way *(torchvision)*
- PyTorch Image Models *(timm)*

<img src="https://upload.wikimedia.org/wikipedia/commons/9/96/Pytorch_logo.png" width="60%">

In [6]:
from torchvision import transforms
import requests

In [ ]:
model_name = "dinov2_vitb14"
model = torch.hub.load('facebookresearch/dinov2', model_name)

# Set model to evaluation mode
model.eval()

## **2. Datasets**
For our experiment we will use the most basic datasets to showcase the different characteristics of feature vectors.

We will need to define a transform to preprocess the images before feeding them into the model. Vision Transformers (like DINOv3) have strict requirements for their input data, as they were pre-trained on a specific data distribution, resolution, and format.

Our transform pipeline must accomplish two critical tasks:

* **1. Resolution Matching:** Our datasets (CIFAR, SVHN) are composed of small 32x32 pixel images. The DINOv3 model was trained on much higher-resolution images (e.g., 224x224). Our transform must resize these small images to match the model's expected input dimensions.
* **2. Data Normalization:** This is the most critical step. The model expects pixel values to be normalized using the *exact* **`mean`** and **`standard deviation`** of the massive dataset it was trained on. If we feed it images with pixel values from 0-255 or 0-1, the resulting feature vectors will be meaningless.

To ensure this is done correctly, we will use the `AutoImageProcessor` associated with the DINOv3 model from Hugging Face. This processor automatically handles the correct resizing, cropping, and normalization, guaranteeing our data is in the perfect format for feature extraction.

In [10]:
processor = AutoImageProcessor.from_pretrained(model_name)
transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

### 2.1 CIFAR-10 and CIFAR-100
The CIFAR-10 and CIFAR-100 datasets are widely used benchmark datasets for image classification tasks in machine learning and computer vision.
- **CIFAR-10**: This dataset consists of 60,000 32x32 color images in 10 different classes, with 6,000 images per class. The classes include airplanes, cars, birds, cats, deer, dogs, frogs, horses, ships, and trucks. The dataset is divided into 50,000 training images and 10,000 test images.
- **CIFAR-100**: Similar to CIFAR-10, this dataset also contains 60,000 32x32 color images, but it is divided into 100 classes with 600 images per class. The 100 classes are grouped into 20 superclasses, providing a more fine-grained classification challenge. Like CIFAR-10, CIFAR-100 is also split into 50,000 training images and 10,000 test images.

In [11]:
dataset_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader_cifar10 = DataLoader(dataset_cifar10, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR10 samples (OOD - CIFAR-10): {len(dataset_cifar10)}")

100%|██████████| 170M/170M [00:03<00:00, 44.0MB/s]


Amount of CIFAR10 samples (OOD - CIFAR-10): 10000


In [12]:
dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
loader_cifar100 = DataLoader(dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(dataset_cifar100)}")

100%|██████████| 169M/169M [00:07<00:00, 21.4MB/s]


Amount of CIFAR100 samples (OOD - CIFAR-100): 10000


### 2.2 SVHN
The Street View House Numbers (SVHN) dataset is a real-world image dataset for developing machine learning and object recognition algorithms with a focus on digit recognition in natural images. It is derived from Google Street View images and contains over 600,000 labeled digits (0-9) cropped from house numbers in street-level photographs.

In [13]:
dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
loader_svhn = DataLoader(dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (Test set): {len(dataset_svhn)}")

100%|██████████| 64.3M/64.3M [00:01<00:00, 35.8MB/s]


Amount of SVHN samples (Test set): 26032


### 2.3 ImageNetV2
ImageNetV2 is an ImageNet subset created to evaluate the robustness and generalization of image classification models. It was designed to address potential biases and overfitting issues present in the original ImageNet dataset by providing a new test set that closely resembles the original data distribution while being collected independently.

In [ ]:
dataset_imagenetv2 = torchvision.datasets.ImageNetV2(root='./data', split='val', download=True, transform=transform)
loader_imagenetv2 = DataLoader(dataset_imagenetv2, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of ImageNetV2 samples (validation set): {len(dataset_imagenetv2)}")

### 2.4 Pokémon Dataset (Bonus)
The Pokémon dataset is a collection of images featuring various Pokémon characters from the popular franchise. It is often used for image classification tasks in machine learning and computer vision.
It uses the original Gen 1 Pokémon (151 characters) as classes.

<img src="https://static0.gamerantimages.com/wordpress/wp-content/uploads/2023/06/pokemon-fusion-art-kanto-gen-1.jpg" width="60%">

In [ ]:
from utils.util import HuggingFaceDatasetWrapper
from datasets import load_dataset

In [21]:
from torch.utils.data import Dataset
from datasets import load_dataset

class HuggingFaceDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform):
        self.hf_dataset = hf_dataset
        self.transform = transform

        if 'labels' in hf_dataset.features:
            self.label_column = 'labels'
        elif 'label' in hf_dataset.features:
            self.label_column = 'label'
        else:
            raise ValueError("Dataset neobsahuje sloupec 'label' ani 'labels'")

        self.classes = hf_dataset.features[self.label_column].names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]

        image = item['image']

        image_tensor = self.transform(image)

        label = item[self.label_column]

        return image_tensor, label

In [ ]:
hf_pokemon_dataset_gen1 = load_dataset(
    "RogerKoala/gen1-pokemon-images",
    split="train"
)

dataset_pokemon_gen1 = HuggingFaceDatasetWrapper(hf_pokemon_dataset_gen1, transform=transform)
loader_pokemon_gen1 = DataLoader(dataset_pokemon_gen1, batch_size=64, shuffle=False, num_workers=2)

## **3. Feature Extraction**

## **4. Mean Feature Vector**

## **5. Principal Component Analysis (PCA)**